## Section 11: Summary and Key Takeaways

### What is GWR?
**Gradient-Weighted Regression (GWR)** is a local regression method that:
- Uses a symbolic **base model** (polynomial, linear, etc.)
- Computes **analytical gradients** at training points
- Makes **localized predictions** using k-nearest neighbors
- Leverages **first-order Taylor expansion** for extrapolation

### Key Advantages of GWR:
1. **Interpretability**: Combines symbolic equations with local predictions
2. **Flexibility**: Adapts locally while maintaining global model structure
3. **Gradient-based**: Uses analytical derivatives for accurate local approximation
4. **Non-parametric locality**: Doesn't assume global linearity or fixed parameters

### How to Use GWR:
```python
# Step 1: Fit a base model and convert to SymPy
model = fit_polynomial_regression(X_train, y_train, degree=2)
sympy_eq, x_symbols = model_to_sympy_equation(model)

# Step 2: Create and fit GWR
gwr = GradientWeightedRegressor(
    sympy_equation=sympy_eq,
    x_symbols=x_symbols,
    n_neighbors=5,  # ← Tune this parameter!
    weighted=True,
)
gwr.fit(X_train, y_train)

# Step 3: Make predictions
y_pred = gwr.predict(X_test)
```

### Tuning Parameters:
- **n_neighbors**: Smaller k = more local, larger k = more global
- **weighted**: Use inverse distance weighting (True) or uniform weighting (False)
- **polynomial_degree**: Higher degree = more flexible base model (but risk of overfitting)

### When to Use GWR:
✓ Non-stationary data with spatial/local patterns
✓ When you have a reasonable symbolic base model
✓ When interpretability matters
✓ Regression problems with local variations

### Explore Further:
Try different combinations of:
- PMLB datasets
- Polynomial degrees
- Number of neighbors
- Custom SymPy equations

Good luck with your GWR experiments!

In [ ]:
# ============================================================================
# STEP 10: Custom SymPy Equation
# ============================================================================

# Example: Define a custom symbolic equation
# Let's say we want to test: y = a*x0^2 + b*x1*x0 + c*x2 + d

print(f"\n{'='*80}")
print(f"CUSTOM SYMPY EQUATION EXAMPLE")
print(f"{'='*80}")

# Create symbols for features
x0, x1, x2 = sp.symbols('x0 x1 x2')

# Define a custom equation
# Example: y = 2*x0^2 + 0.5*x1*x0 - 0.3*x2 + 1.5
custom_equation = 2*x0**2 + 0.5*x1*x0 - 0.3*x2 + 1.5

print(f"\nCustom Equation: {custom_equation}")

# Generate synthetic data that roughly follows this equation
np.random.seed(42)
n_samples = 100
X_synthetic = np.random.randn(n_samples, 3) * 2  # 3 features

# Generate y from the equation plus noise
y_synthetic = np.zeros(n_samples)
for i in range(n_samples):
    subs_dict = {x0: X_synthetic[i, 0], x1: X_synthetic[i, 1], x2: X_synthetic[i, 2]}
    y_synthetic[i] = float(custom_equation.subs(subs_dict)) + np.random.normal(0, 0.5)

X_train_synth, X_test_synth, y_train_synth, y_test_synth = train_test_split(
    X_synthetic, y_synthetic, train_size=0.8, random_state=42
)

print(f"\nSynthetic data shape: {X_synthetic.shape}")
print(f"Train/Test split: {X_train_synth.shape[0]}/{X_test_synth.shape[0]}")

# Since we have the true equation, we can compute perfect predictions as baseline
y_true_train = np.array([float(custom_equation.subs({x0: X_train_synth[i, 0], x1: X_train_synth[i, 1], x2: X_train_synth[i, 2]})) for i in range(len(X_train_synth))])
y_true_test = np.array([float(custom_equation.subs({x0: X_test_synth[i, 0], x1: X_test_synth[i, 1], x2: X_test_synth[i, 2]})) for i in range(len(X_test_synth))])

true_r2 = r2_score(y_test_synth, y_true_test)
print(f"\nTrue equation R² on noisy test data: {true_r2:.6f}")

# Now apply GWR with the custom equation
print(f"\nApplying GWR with custom equation...")
custom_gwr_test = GradientWeightedRegressor(
    sympy_equation=custom_equation,
    x_symbols=[x0, x1, x2],
    n_neighbors=5,
    relative=False,
    weighted=True,
)

custom_gwr_test.fit(X_train_synth, y_train_synth)
y_pred_custom_eq = custom_gwr_test.predict(X_test_synth)

custom_eq_metrics = calculate_metrics(y_test_synth, y_pred_custom_eq, "GWR (Custom Eq)")
print(f"\nGWR Results with custom equation:")
print(f"  R²: {custom_eq_metrics['R²']:.6f}")
print(f"  MSE: {custom_eq_metrics['MSE']:.6f}")
print(f"  MAE: {custom_eq_metrics['MAE']:.6f}")

print(f"\nNote: GWR fits the training data (with noise) and uses local extrapolation,")
print(f"      so it won't perfectly match the true underlying equation.")
print(f"      R² of {custom_eq_metrics['R²']:.6f} reflects prediction accuracy on noisy test data.")

## Section 10: Custom SymPy Equation

You can also define a **custom symbolic equation** directly and test it with GWR!

In [ ]:
# ============================================================================
# STEP 9: CUSTOMIZABLE SECTION - USER INPUT
# ============================================================================

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# OPTION A: Use a Different PMLB Dataset
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# Available datasets (sample list):
# '207_autoPrice', '210_cloud', '215_2dplanes', '218_house_8L', '225_puma8NH'
# '228_elusage', '229_pwLinear', '230_machine_cpu', '294_satellite_image'
# '503_wind', '505_tecator', '560_bodyfat', '561_cpu', '574_house_16H'

CUSTOM_DATASET = "207_autoPrice"  # ← CHANGE THIS to try different datasets

print(f"\n{'='*80}")
print(f"CUSTOM EXPERIMENT: Dataset = {CUSTOM_DATASET}")
print(f"{'='*80}")

try:
    # Load custom dataset
    custom_data_path = f"../data/pmlb/{CUSTOM_DATASET}"
    custom_df = pd.read_csv(f"{custom_data_path}/data.csv")
    
    X_custom = custom_df.iloc[:, :-1].select_dtypes(include=[np.number]).values
    y_custom = custom_df.iloc[:, -1].values
    
    X_train_custom, X_test_custom, y_train_custom, y_test_custom = train_test_split(
        X_custom, y_custom, train_size=0.8, random_state=42
    )
    
    # Fit polynomial model
    CUSTOM_DEGREE = 2  # ← CHANGE THIS to try different degrees (1, 2, 3, ...)
    
    custom_model, _ = fit_polynomial_regression(X_train_custom, y_train_custom, degree=CUSTOM_DEGREE)
    custom_sympy_eq, custom_x_symbols = model_to_sympy_equation(custom_model)
    
    y_pred_custom_base = custom_model.predict(X_test_custom)
    
    # Run GWR
    CUSTOM_K_NEIGHBORS = 5  # ← CHANGE THIS to try different k values
    
    custom_gwr = GradientWeightedRegressor(
        sympy_equation=custom_sympy_eq,
        x_symbols=custom_x_symbols,
        n_neighbors=CUSTOM_K_NEIGHBORS,
        relative=False,
        weighted=True,
    )
    custom_gwr.fit(X_train_custom, y_train_custom)
    y_pred_custom_gwr = custom_gwr.predict(X_test_custom)
    
    # Compare
    print(f"\nDataset: {CUSTOM_DATASET}")
    print(f"Shape: {X_custom.shape}")
    print(f"Polynomial Degree: {CUSTOM_DEGREE}")
    print(f"GWR Neighbors: {CUSTOM_K_NEIGHBORS}")
    
    custom_base_metrics = calculate_metrics(y_test_custom, y_pred_custom_base, "Base Model")
    custom_gwr_metrics = calculate_metrics(y_test_custom, y_pred_custom_gwr, "GWR")
    
    custom_comparison = pd.DataFrame([custom_base_metrics, custom_gwr_metrics])
    print("\n" + custom_comparison.to_string(index=False))
    
    # Improvement
    print(f"\nGWR Improvements:")
    print(f"  R² improvement: {(custom_gwr_metrics['R²'] - custom_base_metrics['R²']):+.6f}")
    print(f"  MSE improvement: {((custom_base_metrics['MSE'] - custom_gwr_metrics['MSE']) / custom_base_metrics['MSE'] * 100):+.2f}%")
    
except FileNotFoundError:
    print(f"Error: Dataset '{CUSTOM_DATASET}' not found. Check the dataset name and path.")
except Exception as e:
    print(f"Error processing dataset: {str(e)}")

## Section 9: User Input - Custom Equations and Datasets

Now it's your turn! Modify the cells below to experiment with:
- **Different PMLB datasets**: Change `DATASET_NAME`
- **Different polynomial degrees**: Adjust `POLYNOMIAL_DEGREE`
- **Different number of neighbors**: Modify `K_NEIGHBORS`
- **Custom SymPy equations**: Define your own symbolic equation

Try different configurations and observe how GWR performance changes!

In [ ]:
# ============================================================================
# STEP 8: Neighbor Sweep - Effect of k on Performance
# ============================================================================

print("=" * 80)
print("EXPLORING THE EFFECT OF NUMBER OF NEIGHBORS (k)")
print("=" * 80)

neighbor_values = [3, 5, 7, 10, 15, 20]
gwr_results = []

for k in neighbor_values:
    gwr_temp = GradientWeightedRegressor(
        sympy_equation=sympy_equation,
        x_symbols=x_symbols,
        n_neighbors=k,
        relative=False,
        weighted=True,
    )
    gwr_temp.fit(X_train, y_train)
    y_pred_temp = gwr_temp.predict(X_test)
    
    metrics_temp = calculate_metrics(y_test, y_pred_temp, f"GWR (k={k})")
    gwr_results.append(metrics_temp)
    
    print(f"k={k:2d}: R²={metrics_temp['R²']:.6f}, MSE={metrics_temp['MSE']:.6f}, MAE={metrics_temp['MAE']:.6f}")

# Create comparison table
gwr_comparison_df = pd.DataFrame(gwr_results)
print("\n" + gwr_comparison_df.to_string(index=False))

# Visualize neighbor effect
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# R² vs k
ax = axes[0]
ax.plot(neighbor_values, [m['R²'] for m in gwr_results], 'o-', linewidth=2, markersize=8, color='green', label='GWR')
ax.axhline(y=metrics_base['R²'], color='blue', linestyle='--', linewidth=2, label='Base Model')
ax.set_xlabel('Number of Neighbors (k)')
ax.set_ylabel('R² Score')
ax.set_title('R² vs Number of Neighbors')
ax.grid(True, alpha=0.3)
ax.legend()

# MSE vs k
ax = axes[1]
ax.plot(neighbor_values, [m['MSE'] for m in gwr_results], 'o-', linewidth=2, markersize=8, color='orange', label='GWR')
ax.axhline(y=metrics_base['MSE'], color='blue', linestyle='--', linewidth=2, label='Base Model')
ax.set_xlabel('Number of Neighbors (k)')
ax.set_ylabel('Mean Squared Error')
ax.set_title('MSE vs Number of Neighbors')
ax.grid(True, alpha=0.3)
ax.legend()

# MAE vs k
ax = axes[2]
ax.plot(neighbor_values, [m['MAE'] for m in gwr_results], 'o-', linewidth=2, markersize=8, color='red', label='GWR')
ax.axhline(y=metrics_base['MAE'], color='blue', linestyle='--', linewidth=2, label='Base Model')
ax.set_xlabel('Number of Neighbors (k)')
ax.set_ylabel('Mean Absolute Error')
ax.set_title('MAE vs Number of Neighbors')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print("\n✓ Neighbor sweep visualization completed!")

## Section 8: Exploring the Effect of Number of Neighbors

Let's see how the number of neighbors (k) affects GWR performance.

In [ ]:
# ============================================================================
# STEP 7: Visualizations
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Base Model vs GWR Comparison', fontsize=16, fontweight='bold')

# Plot 1: Actual vs Predicted (Base Model)
ax = axes[0, 0]
ax.scatter(y_test, y_pred_test_base, alpha=0.6, s=50, label='Base Model')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual Target Value')
ax.set_ylabel('Predicted Value')
ax.set_title(f'Base Model (Poly deg={POLYNOMIAL_DEGREE}) - R²={metrics_base["R²"]:.4f}')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Actual vs Predicted (GWR)
ax = axes[0, 1]
ax.scatter(y_test, y_pred_gwr_library, alpha=0.6, s=50, color='green', label='GWR')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual Target Value')
ax.set_ylabel('Predicted Value')
ax.set_title(f'GWR (k={N_NEIGHBORS}) - R²={metrics_gwr["R²"]:.4f}')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Residuals (Base Model)
ax = axes[1, 0]
ax.scatter(y_pred_test_base, residuals_base, alpha=0.6, s=50)
ax.axhline(y=0, color='r', linestyle='--', lw=2)
ax.set_xlabel('Predicted Value')
ax.set_ylabel('Residual (Actual - Predicted)')
ax.set_title(f'Base Model Residuals (Std={np.std(residuals_base):.4f})')
ax.grid(True, alpha=0.3)

# Plot 4: Residuals (GWR)
ax = axes[1, 1]
ax.scatter(y_pred_gwr_library, residuals_gwr, alpha=0.6, s=50, color='green')
ax.axhline(y=0, color='r', linestyle='--', lw=2)
ax.set_xlabel('Predicted Value')
ax.set_ylabel('Residual (Actual - Predicted)')
ax.set_title(f'GWR Residuals (Std={np.std(residuals_gwr):.4f})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualization completed!")


# Plot 2: Error distribution comparison
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

errors_base = np.abs(residuals_base)
errors_gwr = np.abs(residuals_gwr)

ax.hist(errors_base, bins=30, alpha=0.6, label=f'Base Model (Mean={np.mean(errors_base):.4f})', color='blue')
ax.hist(errors_gwr, bins=30, alpha=0.6, label=f'GWR (Mean={np.mean(errors_gwr):.4f})', color='green')
ax.set_xlabel('Absolute Error')
ax.set_ylabel('Frequency')
ax.set_title('Absolute Error Distribution: Base Model vs GWR')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Error distribution plot completed!")

## Section 7: Visualize Results

Let's create visualizations to compare predictions and understand GWR's behavior.

In [ ]:
# ============================================================================
# STEP 6: Compare Standard Regression vs GWR
# ============================================================================

def calculate_metrics(y_true, y_pred, model_name=""):
    """Calculate comprehensive accuracy metrics."""
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mse)
    
    # Mean Absolute Percentage Error (MAPE)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-10))) * 100
    
    return {
        'Model': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'MAPE': mape,
    }


print("=" * 80)
print("ACCURACY COMPARISON: BASE MODEL vs GWR")
print("=" * 80)

# Calculate metrics for both methods
metrics_base = calculate_metrics(y_test, y_pred_test_base, "Polynomial Regression")
metrics_gwr = calculate_metrics(y_test, y_pred_gwr_library, "GWR (k=5)")

# Create comparison dataframe
comparison_df = pd.DataFrame([metrics_base, metrics_gwr])
print("\n" + comparison_df.to_string(index=False))

# Calculate improvements
print("\n" + "=" * 80)
print("GWR IMPROVEMENTS OVER BASE MODEL")
print("=" * 80)

improvement_mse = ((metrics_base['MSE'] - metrics_gwr['MSE']) / metrics_base['MSE']) * 100
improvement_mae = ((metrics_base['MAE'] - metrics_gwr['MAE']) / metrics_base['MAE']) * 100
improvement_r2 = metrics_gwr['R²'] - metrics_base['R²']
improvement_mape = ((metrics_base['MAPE'] - metrics_gwr['MAPE']) / metrics_base['MAPE']) * 100

print(f"MSE:  {improvement_mse:+.2f}% {'✓' if improvement_mse > 0 else '✗'}")
print(f"RMSE: {(np.sqrt(metrics_base['MSE']) - np.sqrt(metrics_gwr['MSE'])) / np.sqrt(metrics_base['MSE']) * 100:+.2f}% {'✓' if improvement_mse > 0 else '✗'}")
print(f"MAE:  {improvement_mae:+.2f}% {'✓' if improvement_mae > 0 else '✗'}")
print(f"R²:   {improvement_r2:+.6f} {'✓' if improvement_r2 > 0 else '✗'}")
print(f"MAPE: {improvement_mape:+.2f}% {'✓' if improvement_mape > 0 else '✗'}")

print("\n" + "=" * 80)
print("RESIDUALS ANALYSIS")
print("=" * 80)

residuals_base = y_test - y_pred_test_base
residuals_gwr = y_test - y_pred_gwr_library

print(f"\nBase Model Residuals:")
print(f"  Mean: {np.mean(residuals_base):.6f}")
print(f"  Std:  {np.std(residuals_base):.6f}")
print(f"  Min:  {np.min(residuals_base):.6f}")
print(f"  Max:  {np.max(residuals_base):.6f}")

print(f"\nGWR Residuals:")
print(f"  Mean: {np.mean(residuals_gwr):.6f}")
print(f"  Std:  {np.std(residuals_gwr):.6f}")
print(f"  Min:  {np.min(residuals_gwr):.6f}")
print(f"  Max:  {np.max(residuals_gwr):.6f}")

## Section 6: Compare Accuracy Metrics: Standard Regression vs GWR

Now let's compare the performance of the base polynomial model with GWR. We'll use multiple metrics to assess accuracy.

In [ ]:
# ============================================================================
# STEP 5: GWR Library Implementation
# ============================================================================

print("Initializing GWR library implementation...")

# Create and fit GWR model
gwr_model = GradientWeightedRegressor(
    sympy_equation=sympy_equation,
    x_symbols=x_symbols,
    n_neighbors=N_NEIGHBORS,
    relative=False,  # Use absolute neighbor selection
    weighted=True,   # Use inverse distance weighting
)

print("Fitting GWR model on training data...")
gwr_model.fit(X_train, y_train)

print("Making predictions on test data...")
y_pred_gwr_library = gwr_model.predict(X_test)

print(f"✓ GWR library predictions completed!")
print(f"  Shape: {y_pred_gwr_library.shape}")
print(f"  First 10 predictions: {y_pred_gwr_library[:10]}")

# Compare with manual implementation
difference = np.abs(y_pred_gwr_manual - y_pred_gwr_library)
max_diff = np.max(difference)
mean_diff = np.mean(difference)
print(f"\nComparison with manual implementation:")
print(f"  Max difference: {max_diff:.2e}")
print(f"  Mean difference: {mean_diff:.2e}")
print(f"  (Small differences are expected due to floating point precision)")

## Section 5: Using the GWR Library (Built-in Implementation)

The `GradientWeightedRegressor` class from the gwr library provides an optimized implementation. Let's use it alongside our manual implementation.

In [ ]:
# ============================================================================
# STEP 4: Complete GWR Function (Vectorized)
# ============================================================================

def gwr_predict_vectorized(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    sympy_eq: sp.Expr,
    x_symbols: list,
    n_neighbors: int = 5,
    weighted: bool = True,
    verbose: bool = False,
) -> np.ndarray:
    """
    Vectorized Gradient-Weighted Regression prediction.
    
    Parameters
    ----------
    X_train : np.ndarray
        Training features (reference points)
    y_train : np.ndarray
        Training targets
    X_test : np.ndarray
        Test features (query points)
    sympy_eq : sp.Expr
        SymPy symbolic equation of base model
    x_symbols : list
        Ordered list of SymPy symbols for features
    n_neighbors : int
        Number of nearest neighbors to use
    weighted : bool
        Whether to use inverse-distance weighting
    verbose : bool
        Print progress information
        
    Returns
    -------
    predictions : np.ndarray
        GWR predictions for test points
    """
    
    X_train = np.asarray(X_train, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=np.float64)
    X_test = np.asarray(X_test, dtype=np.float64)
    
    # Step 1: Compute gradients at all training points
    if verbose:
        print("Computing gradients at training points...")
    gradients = evaluate_gradients(sympy_eq, x_symbols, X_train)
    
    # Step 2: Build KD-tree
    if verbose:
        print("Building KD-tree...")
    kdtree = cKDTree(X_train)
    
    # Step 3: Find neighbors for all test points
    if verbose:
        print(f"Finding {n_neighbors} nearest neighbors for {X_test.shape[0]} test points...")
    distances, indices = kdtree.query(X_test, k=n_neighbors)
    
    # Handle case where k=1
    if n_neighbors == 1:
        distances = distances[:, np.newaxis]
        indices = indices[:, np.newaxis]
    
    # Step 4: Calculate weights
    epsilon = 1e-6
    weights = 1.0 / (distances + epsilon)
    weights = weights / weights.sum(axis=1, keepdims=True)
    
    # Step 5: Get neighbor data
    neighbor_targets = y_train[indices]  # (n_test, k)
    neighbor_features = X_train[indices]  # (n_test, k, n_features)
    neighbor_gradients = gradients[indices]  # (n_test, k, n_features)
    
    # Step 6: Calculate directional derivative predictions
    # directions: (n_test, k, n_features)
    directions = X_test[:, np.newaxis, :] - neighbor_features
    
    # Vectorized dot product for all test points and neighbors
    directions_flat = directions.reshape(-1, directions.shape[-1])
    gradients_flat = neighbor_gradients.reshape(-1, neighbor_gradients.shape[-1])
    changes_flat = np.einsum("ij,ij->i", directions_flat, gradients_flat)
    changes = changes_flat.reshape(directions.shape[0], directions.shape[1])
    
    # Predicted values from each neighbor
    predictions_per_neighbor = neighbor_targets + changes
    
    # Step 7: Combine predictions
    if weighted:
        predictions = np.sum(predictions_per_neighbor * weights, axis=1)
    else:
        predictions = predictions_per_neighbor.mean(axis=1)
    
    return predictions


# Test the function
print("Running vectorized GWR on full test set...")
y_pred_gwr_manual = gwr_predict_vectorized(
    X_train, y_train, X_test, sympy_equation, x_symbols,
    n_neighbors=N_NEIGHBORS, weighted=True, verbose=True
)

print(f"✓ GWR predictions completed!")
print(f"  Shape: {y_pred_gwr_manual.shape}")
print(f"  First 10 predictions: {y_pred_gwr_manual[:10]}")

## Section 4: Full GWR Function (Vectorized Implementation)

Now let's implement the complete GWR function in a vectorized manner for efficiency.

In [ ]:
# ============================================================================
# STEP 3: GWR STEP-BY-STEP IMPLEMENTATION
# ============================================================================

print("=" * 70)
print("GWR STEP 1: Compute Gradients at All Training Points")
print("=" * 70)

# Compute partial derivatives at each training point
all_gradients = evaluate_gradients(sympy_equation, x_symbols, X_train)
print(f"✓ Gradients computed for {X_train.shape[0]} training points")
print(f"✓ Gradient matrix shape: {all_gradients.shape}")
print(f"  - Rows: training samples")
print(f"  - Columns: partial derivatives for each feature")
print(f"\nFirst 5 gradients (sample):")
print(all_gradients[:5])

print("\n" + "=" * 70)
print("GWR STEP 2: Build KD-Tree for Efficient Neighbor Search")
print("=" * 70)

# Build KD-tree on training points
kdtree = cKDTree(X_train)
print(f"✓ KD-Tree built for {X_train.shape[0]} training points with {X_train.shape[1]} dimensions")

print("\n" + "=" * 70)
print("GWR STEP 3: Find Neighbors and Calculate Weights")
print("=" * 70)

# For demonstration, look at the first test point
N_NEIGHBORS = 5
test_point_idx = 0
test_point = X_test[test_point_idx:test_point_idx+1]

# Find k nearest neighbors
distances, neighbor_indices = kdtree.query(test_point, k=N_NEIGHBORS)
distances = distances[0]
neighbor_indices = neighbor_indices[0]

print(f"\nTest point #{test_point_idx}:")
print(f"  Features: {test_point[0]}")
print(f"\nNearest {N_NEIGHBORS} neighbors:")
print(f"{'Index':<8} {'Distance':<15} {'y_train':<15}")
print("-" * 38)
for i, (idx, dist) in enumerate(zip(neighbor_indices, distances)):
    print(f"{idx:<8} {dist:<15.6f} {y_train[idx]:<15.6f}")

# Calculate inverse distance weights
print(f"\nSpatial Weights (inverse distance):")
epsilon = 1e-6
weights = 1.0 / (distances + epsilon)
weights = weights / weights.sum()

print(f"{'Index':<8} {'Distance':<15} {'Weight':<15}")
print("-" * 38)
for idx, dist, weight in zip(neighbor_indices, distances, weights):
    print(f"{idx:<8} {dist:<15.6f} {weight:<15.6f}")

print("\n" + "=" * 70)
print("GWR STEP 4: First-Order Taylor Expansion Prediction")
print("=" * 70)

# Get gradients at neighbor points
neighbor_gradients = all_gradients[neighbor_indices]
neighbor_targets = y_train[neighbor_indices]
neighbor_features = X_train[neighbor_indices]

print(f"\nPrediction formula: ŷ = y_neighbor + ∇f(x_neighbor) · (x_test - x_neighbor)")
print(f"\nCalculations for test point #{test_point_idx}:")
print(f"{'Neighbor':<10} {'y_neighbor':<15} {'Gradient·Direction':<20} {'ŷ_neighbor':<15}")
print("-" * 60)

predictions_from_neighbors = []
for i, (idx, y_ref, grad, x_ref, weight) in enumerate(zip(
    neighbor_indices, neighbor_targets, neighbor_gradients, neighbor_features, weights
)):
    # Calculate direction from neighbor to test point
    direction = test_point[0] - x_ref
    # Change in target using gradient
    change = np.dot(grad, direction)
    # Predicted value at test point from this neighbor
    y_pred_from_neighbor = y_ref + change
    predictions_from_neighbors.append(y_pred_from_neighbor)
    
    print(f"{idx:<10} {y_ref:<15.6f} {change:<20.6f} {y_pred_from_neighbor:<15.6f}")

# Weighted average of predictions
predictions_from_neighbors = np.array(predictions_from_neighbors)
y_gwr_single = np.sum(predictions_from_neighbors * weights)

print(f"\n{'Weighted Average (GWR):':<50} {y_gwr_single:.6f}")
print(f"{'Actual target value:':<50} {y_test[test_point_idx]:.6f}")
print(f"{'Error (GWR):':<50} {abs(y_gwr_single - y_test[test_point_idx]):.6f}")
print(f"{'Base model prediction:':<50} {y_pred_test_base[test_point_idx]:.6f}")
print(f"{'Error (Base):':<50} {abs(y_pred_test_base[test_point_idx] - y_test[test_point_idx]):.6f}")

## Section 3: GWR Step-by-Step (Manual Implementation)

Let's implement GWR step-by-step to understand the methodology:

### GWR Algorithm:
1. **Compute Gradients**: Calculate partial derivatives of the equation at each training point
2. **Build KD-Tree**: Create spatial index of training points for efficient neighbor search
3. **For each test point**:
   - Find k-nearest training neighbors
   - Calculate spatial weights (inverse distance)
   - Use first-order Taylor expansion: $\hat{y}_{test} \approx y_{ref} + \nabla f(x_{ref}) \cdot (x_{test} - x_{ref})$
   - Weight and combine predictions from all k neighbors

In [ ]:
# ============================================================================
# STEP 2: Fit Base Polynomial Model
# ============================================================================

# Define polynomial degree (try 2 or 3 for complexity vs. accuracy tradeoff)
POLYNOMIAL_DEGREE = 2

print(f"Fitting polynomial regression (degree={POLYNOMIAL_DEGREE})...")
model, y_pred_train_base = fit_polynomial_regression(
    X_train, y_train, X_test=None, degree=POLYNOMIAL_DEGREE
)

# Get predictions on test set
y_pred_test_base = model.predict(X_test)

print(f"✓ Base model fitted!")

# Convert the fitted model to a SymPy symbolic equation
print("\nConverting model to SymPy equation...")
sympy_equation, x_symbols = model_to_sympy_equation(model)

print(f"✓ SymPy equation created with {len(x_symbols)} features")
print(f"\nBase equation (truncated):")
eq_str = str(sympy_equation)
if len(eq_str) > 300:
    print(eq_str[:300] + "...")
else:
    print(eq_str)

# Calculate training metrics for base model
y_pred_train_base_full = model.predict(X_train)
base_train_r2 = r2_score(y_train, y_pred_train_base_full)
base_train_mse = mean_squared_error(y_train, y_pred_train_base_full)
base_train_mae = mean_absolute_error(y_train, y_pred_train_base_full)

base_test_r2 = r2_score(y_test, y_pred_test_base)
base_test_mse = mean_squared_error(y_test, y_pred_test_base)
base_test_mae = mean_absolute_error(y_test, y_pred_test_base)

print(f"\n{'='*60}")
print(f"BASE MODEL METRICS")
print(f"{'='*60}")
print(f"Train - R²: {base_train_r2:.6f}, MSE: {base_train_mse:.6f}, MAE: {base_train_mae:.6f}")
print(f"Test  - R²: {base_test_r2:.6f}, MSE: {base_test_mse:.6f}, MAE: {base_test_mae:.6f}")

## Section 2: Define and Fit Base Model

We'll fit a **polynomial regression model** as our base model. This model will be converted to a SymPy equation, allowing us to compute analytical gradients at each reference point.

The polynomial coefficients define our closed-form equation: $\hat{y} = \beta_0 + \sum \beta_i x_i + \sum \beta_{ij} x_i x_j + ...$

In [ ]:
# ============================================================================
# STEP 1: Load PMLB Dataset and Split into Train/Test
# ============================================================================

# Select a PMLB dataset (you can change this)
DATASET_NAME = "207_autoPrice"  # Change this to try different datasets
DATA_PATH = f"../data/pmlb/{DATASET_NAME}"

# Load data
print(f"Loading dataset: {DATASET_NAME}")
df = pd.read_csv(f"{DATA_PATH}/data.csv")

print(f"\nDataset shape: {df.shape}")
print(f"First few rows:")
print(df.head())

# Extract features and target
# Assume the last column is the target
X = df.iloc[:, :-1].select_dtypes(include=[np.number]).values
y = df.iloc[:, -1].values

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature names: {list(df.columns[:-1])}")
print(f"Target name: {df.columns[-1]}")

# Split into train/test (80/20 split)
train_frac = 0.8
random_state = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=train_frac, random_state=random_state
)

print(f"\n✓ Train set: {X_train.shape[0]} samples")
print(f"✓ Test set: {X_test.shape[0]} samples")

## Section 1: Load PMLB Dataset and Split Data

In this section, we'll load a regression dataset from PMLB. The training data will serve as **reference points** for GWR. The test data will be used to evaluate both standard regression and GWR predictions.

In [ ]:
# Import Required Libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.spatial import cKDTree
import sympy as sp
import warnings
warnings.filterwarnings('ignore')

# Add the project root to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import GWR modules
from gwr.core import GradientWeightedRegressor
from gwr.models import (
    fit_polynomial_regression,
    fit_linear_regression,
    model_to_sympy_equation,
)
from gwr.gradients import evaluate_gradients
from gwr.utils import compute_all_metrics, load_dataset

print("✓ All libraries imported successfully!")
print(f"✓ GWR modules loaded from: {os.path.abspath('.')}")

# Geographically Weighted Regression (GWR) for PMLB Regression

This notebook demonstrates **Gradient-Weighted Regression (GWR)**, a method that uses analytical gradients from a base model to make localized predictions at query points. 

## What is GWR?
GWR is a non-parametric regression technique that:
1. Starts with a **closed-form base model** (polynomial, linear, etc.)
2. Computes **analytical gradients** at every reference (training) point
3. For each test point, finds the **k-nearest reference points**
4. Uses **weighted first-order Taylor expansion** to predict the target value at that test point

This approach combines the interpretability of symbolic equations with the flexibility of local, distance-weighted predictions.

## Key Concepts
- **Reference Points**: Training data used as anchor points for local extrapolation
- **Adaptive Bandwidth**: Number of neighbors (k) determines the locality of predictions
- **Spatial Weights**: Inverse distance weighting (or uniform) combines neighbor predictions
- **Gradient-Based Extrapolation**: First-order Taylor expansion from neighbor gradients

Let's walk through a complete GWR pipeline on PMLB datasets!